In [6]:
import os
import requests

BEA_BASE_URL = "https://apps.bea.gov/api/data"


def _load_env_local(path=".env.local"):
    env = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, value = line.partition("=")
                env[key.strip()] = value.strip()
    return env


BEA_API_KEY = os.environ.get("BEA_API_KEY") or _load_env_local().get("BEA_API_KEY")


def get_raw_bea_real_gdp_by_industry_state_json(
    geo_fips="STATE",
    year="ALL",
    line_code="1",
    table_name="SQGDP9",
):
    """
    Fetch raw JSON directly from the BEA Regional API for quarterly real GDP
    by state by industry.

    TableName SQGDP9 = Real GDP by state (millions of chained dollars), by
    industry, quarterly. BEA only allows LineCode="ALL" (every industry) when
    GeoFips is a single specific geography (e.g. one state FIPS code) - not
    the "STATE" aggregate keyword. So GeoFips="STATE" (all states) must be
    paired with one specific line_code per call; loop over line codes to
    cover every industry.

    line_code defaults to "1" (the first/total industry line) rather than
    "ALL" so that the default call is a single valid request on its own -
    geo_fips="1" and year="1" were tried too, but GeoFips rejects a bare "1"
    (APIErrorCode 40, "Value of GeoFips is invalid") and Year="1" returns 0
    rows, so those two keep their working defaults ("STATE" / "ALL").

    Returns the original API response without converting it to a DataFrame
    or writing CSV/JSON files.
    """
    params = {
        "UserID": BEA_API_KEY,
        "method": "GetData",
        "datasetname": "Regional",
        "TableName": table_name,
        "LineCode": line_code,
        "GeoFips": geo_fips,
        "Year": year,
        "ResultFormat": "JSON",
    }

    response = requests.get(BEA_BASE_URL, params=params, timeout=60)

    if not response.ok:
        print("FAILED URL:", response.url)
        print("STATUS:", response.status_code)
        print("BODY:", response.text[:500])

    response.raise_for_status()
    return response.json()


def get_bea_regional_line_codes(table_name="SQGDP9"):
    """
    Fetch every valid LineCode (industry) for a Regional table via
    GetParameterValuesFiltered. Returns a list of {"Key": ..., "Desc": ...}.
    """
    params = {
        "UserID": BEA_API_KEY,
        "method": "GetParameterValuesFiltered",
        "datasetname": "Regional",
        "TargetParameter": "LineCode",
        "TableName": table_name,
        "ResultFormat": "JSON",
    }

    response = requests.get(BEA_BASE_URL, params=params, timeout=60)

    if not response.ok:
        print("FAILED URL:", response.url)
        print("STATUS:", response.status_code)
        print("BODY:", response.text[:500])

    response.raise_for_status()
    return response.json()["BEAAPI"]["Results"]["ParamValue"]

In [7]:
import pandas as pd

# BEA rejects LineCode="ALL" combined with GeoFips="STATE" in a single call
# (APIErrorCode 41 - "Only one geography can be submitted when LineCode
# value= ALL"). So instead: fetch every industry's LineCode, then loop -
# one GetData call per industry, each pulling every state and every quarter
# back to 2005 - and concatenate into one long DataFrame.
line_codes = get_bea_regional_line_codes()

frames = []
for lc in line_codes:
    resp = get_raw_bea_real_gdp_by_industry_state_json(
        geo_fips="STATE",
        year="ALL",
        line_code=lc["Key"],
    )
    frames.append(pd.DataFrame(resp["BEAAPI"]["Results"]["Data"]))

df = pd.concat(frames, ignore_index=True)
df["DataValue"] = pd.to_numeric(df["DataValue"], errors="coerce")

print(f"{len(df):,} rows | states: {df['GeoName'].nunique()} | industries: {df['Code'].nunique()} | quarters: {df['TimePeriod'].nunique()}")
df.head()

137,700 rows | states: 60 | industries: 27 | quarters: 85


,Code,GeoFips,GeoName,TimePeriod,CL_UNIT,UNIT_MULT,DataValue,NoteRef
0,SQGDP9-1,00000,United States,2005Q1,Millions of chained 2017 dollars,6,15844727.0,NaN
1,SQGDP9-1,00000,United States,2005Q2,Millions of chained 2017 dollars,6,15922782.0,NaN
2,SQGDP9-1,00000,United States,2005Q3,Millions of chained 2017 dollars,6,16047587.0,NaN
3,SQGDP9-1,00000,United States,2005Q4,Millions of chained 2017 dollars,6,16136734.0,NaN
4,SQGDP9-1,00000,United States,2006Q1,Millions of chained 2017 dollars,6,16353835.0,NaN


In [8]:
# Single-call example using the function's defaults (GeoFips="STATE",
# Year="ALL", LineCode="1", TableName="SQGDP9") - all states, full history,
# one industry line, in one request. Download the entries as a CSV.
single_resp = get_raw_bea_real_gdp_by_industry_state_json()

beaapi = single_resp["BEAAPI"]
if "Error" in beaapi:
    raise RuntimeError(beaapi["Error"])

single_df = pd.DataFrame(beaapi["Results"]["Data"])
single_df["DataValue"] = pd.to_numeric(single_df["DataValue"], errors="coerce")

csv_path = "bea_gdp_industry_state.csv"
single_df.to_csv(csv_path, index=False)

print(f"Saved {len(single_df):,} rows to {csv_path}")
single_df.head()

Saved 5,100 rows to bea_gdp_industry_state.csv


,Code,GeoFips,GeoName,TimePeriod,CL_UNIT,UNIT_MULT,DataValue
0,SQGDP9-1,00000,United States,2005Q1,Millions of chained 2017 dollars,6,15844727.0
1,SQGDP9-1,00000,United States,2005Q2,Millions of chained 2017 dollars,6,15922782.0
2,SQGDP9-1,00000,United States,2005Q3,Millions of chained 2017 dollars,6,16047587.0
3,SQGDP9-1,00000,United States,2005Q4,Millions of chained 2017 dollars,6,16136734.0
4,SQGDP9-1,00000,United States,2006Q1,Millions of chained 2017 dollars,6,16353835.0


In [9]:
beaapi


{'Request': {'RequestParam': [{'ParameterName': 'USERID',
    'ParameterValue': '21549F69-BBAE-44C3-BFCE-EDBABEB88F8C'},
   {'ParameterName': 'METHOD', 'ParameterValue': 'GETDATA'},
   {'ParameterName': 'DATASETNAME', 'ParameterValue': 'REGIONAL'},
   {'ParameterName': 'TABLENAME', 'ParameterValue': 'SQGDP9'},
   {'ParameterName': 'LINECODE', 'ParameterValue': '1'},
   {'ParameterName': 'GEOFIPS', 'ParameterValue': 'STATE'},
   {'ParameterName': 'YEAR', 'ParameterValue': 'ALL'},
   {'ParameterName': 'RESULTFORMAT', 'ParameterValue': 'JSON'}]},
 'Results': {'Statistic': 'Real GDP by state: All industry total',
  'UnitOfMeasure': 'Millions of chained 2017 dollars',
  'PublicTable': 'SQGDP9 Real GDP by state',
  'UTCProductionTime': '2026-08-04T01:29:47.900',
  'NoteRef': ' ',
  'Dimensions': [{'Name': 'Code', 'DataType': 'string', 'IsValue': '0'},
   {'Name': 'GeoFips', 'DataType': 'string', 'IsValue': '0'},
   {'Name': 'GeoName', 'DataType': 'string', 'IsValue': '0'},
   {'Name': 'TimeP

In [10]:
df = pd.read_csv('bea_gdp_industry_state.csv')

In [11]:
df

,Code,GeoFips,GeoName,TimePeriod,CL_UNIT,UNIT_MULT,DataValue
0,SQGDP9-1,0,United States,2005Q1,Millions of chained 2017 dollars,6,15844727.0
1,SQGDP9-1,0,United States,2005Q2,Millions of chained 2017 dollars,6,15922782.0
2,SQGDP9-1,0,United States,2005Q3,Millions of chained 2017 dollars,6,16047587.0
3,SQGDP9-1,0,United States,2005Q4,Millions of chained 2017 dollars,6,16136734.0
4,SQGDP9-1,0,United States,2006Q1,Millions of chained 2017 dollars,6,16353835.0
...,...,...,...,...,...,...,...
5095,SQGDP9-1,98000,Far West,2025Q1,Millions of chained 2017 dollars,6,4674104.5
5096,SQGDP9-1,98000,Far West,2025Q2,Millions of chained 2017 dollars,6,4719587.8
5097,SQGDP9-1,98000,Far West,2025Q3,Millions of chained 2017 dollars,6,4769215.0
5098,SQGDP9-1,98000,Far West,2025Q4,Millions of chained 2017 dollars,6,4776526.7


In [17]:
df["UNIT_MULT"].unique()

array([6])

In [19]:
pip install beaapi

Note: you may need to restart the kernel to use updated packages.


# line code 
# fips code

In [22]:
from __future__ import annotations

import os

import beaapi
import pandas as pd
from dotenv import load_dotenv


load_dotenv()


def download_sqgdp9_reference(
    api_key: str,
    output_directory: str = ".",
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Download the valid SQGDP9 geography codes and industry line codes.

    Produces:
        sqgdp9_geographies.csv
        sqgdp9_industries.csv
        sqgdp9_geo_industry_grid.csv
    """

    # Geographic codes valid for SQGDP9.
    geographies = beaapi.get_parameter_values_filtered(
        api_key,
        "Regional",
        targetparameter="GeoFips",
        TableName="SQGDP9",
    )

    geographies = (
        geographies.rename(
            columns={
                "Key": "GeoFips",
                "Desc": "GeoName",
            }
        )[["GeoFips", "GeoName"]]
        .copy()
    )

    # Preserve leading zeroes.
    geographies["GeoFips"] = (
        geographies["GeoFips"]
        .astype("string")
        .str.strip()
    )

    geographies["GeoName"] = (
        geographies["GeoName"]
        .astype("string")
        .str.strip()
    )

    # Industry line codes valid for SQGDP9.
    industries = beaapi.get_parameter_values_filtered(
        api_key,
        "Regional",
        targetparameter="LineCode",
        TableName="SQGDP9",
    )

    industries = (
        industries.rename(
            columns={
                "Key": "LineCode",
                "Desc": "IndustryName",
            }
        )[["LineCode", "IndustryName"]]
        .copy()
    )

    industries["LineCode"] = (
        industries["LineCode"]
        .astype("string")
        .str.strip()
    )

    industries["IndustryName"] = (
        industries["IndustryName"]
        .astype("string")
        .str.replace(
            r"^\[SQGDP9\]\s*",
            "",
            regex=True,
        )
        .str.strip()
    )

    # Construct every possible geography × industry pairing.
    reference_grid = geographies.merge(
        industries,
        how="cross",
    )

    geographies.to_csv(
        f"{output_directory}/sqgdp9_geographies.csv",
        index=False,
    )

    industries.to_csv(
        f"{output_directory}/sqgdp9_industries.csv",
        index=False,
    )

    reference_grid.to_csv(
        f"{output_directory}/sqgdp9_geo_industry_grid.csv",
        index=False,
    )

    return geographies, industries, reference_grid


geographies, industries, reference_grid = (
    download_sqgdp9_reference(BEA_API_KEY)
)

print("Geographies:")
print(geographies.head())

print("\nIndustries:")
print(industries.head())

print("\nCombined reference grid:")
print(reference_grid.head())

Geographies:
  GeoFips        GeoName
0   00000  United States
1   01000        Alabama
2   02000         Alaska
3   04000        Arizona
4   05000       Arkansas

Industries:
  LineCode                                       IndustryName
0        1              Real GDP by state: All industry total
1       10                  Real GDP by state: Utilities (22)
2       11               Real GDP by state: Construction (23)
3       12           Real GDP by state: Manufacturing (31-33)
4       13  Real GDP by state: Durable goods manufacturing...

Combined reference grid:
  GeoFips        GeoName LineCode  \
0   00000  United States        1   
1   00000  United States       10   
2   00000  United States       11   
3   00000  United States       12   
4   00000  United States       13   

                                        IndustryName  
0              Real GDP by state: All industry total  
1                  Real GDP by state: Utilities (22)  
2               Real GDP by state: Con

In [23]:
industries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   LineCode      27 non-null     string
 1   IndustryName  27 non-null     string
dtypes: string(2)
memory usage: 564.0 bytes


In [24]:
industries

,LineCode,IndustryName
0,1,Real GDP by state: All industry total
1,10,Real GDP by state: Utilities (22)
2,11,Real GDP by state: Construction (23)
3,12,Real GDP by state: Manufacturing (31-33)
4,13,Real GDP by state: Durable goods manufacturing...
5,2,Real GDP by state: Private industries
6,25,Real GDP by state: Nondurable goods manufactur...
7,3,"Real GDP by state: Agriculture, forestry, fish..."
8,34,Real GDP by state: Wholesale trade (42)
9,35,Real GDP by state: Retail trade (44-45)


In [ ]:
from __future__ import annotations

import time

import beaapi
import pandas as pd


def download_all_sqgdp9_data(
    api_key: str,
    years: str | list[int],
    output_file: str = "sqgdp9_all_states_all_industries.csv",
) -> pd.DataFrame:
    """
    Download SQGDP9 observations for every available industry
    and all states for the requested years.

    Examples:
        years="2025"
        years=[2024, 2025]
        years="LAST5"
    """

    if isinstance(years, list):
        year_parameter = ",".join(str(year) for year in years)
    else:
        year_parameter = str(years)

    industries = beaapi.get_parameter_values_filtered(
        api_key,
        "Regional",
        targetparameter="LineCode",
        TableName="SQGDP9",
    )

    industries = industries.rename(
        columns={
            "Key": "LineCode",
            "Desc": "IndustryName",
        }
    )

    industries["LineCode"] = (
        industries["LineCode"]
        .astype("string")
        .str.strip()
    )

    industries["IndustryName"] = (
        industries["IndustryName"]
        .astype("string")
        .str.replace(
            r"^\[SQGDP9\]\s*",
            "",
            regex=True,
        )
        .str.strip()
    )

    result_frames: list[pd.DataFrame] = []

    for industry in industries.itertuples(index=False):
        print(
            f"Downloading line {industry.LineCode}: "
            f"{industry.IndustryName}"
        )

        data = beaapi.get_data(
            api_key,
            datasetname="Regional",
            TableName="SQGDP9",
            LineCode=str(industry.LineCode),
            GeoFips="STATE",
            Year=year_parameter,
        )

        if data.empty:
            continue

        # Explicitly attach the metadata used for the query.
        data["LineCode"] = str(industry.LineCode)
        data["IndustryName"] = industry.IndustryName

        result_frames.append(data)

        # Gentle pacing between requests.
        time.sleep(0.2)

    if not result_frames:
        raise RuntimeError(
            "No data was returned for the requested years."
        )

    combined = pd.concat(
        result_frames,
        ignore_index=True,
    )

    # Preserve geographic codes as strings.
    if "GeoFips" in combined.columns:
        combined["GeoFips"] = (
            combined["GeoFips"]
            .astype("string")
            .str.strip()
        )

    # Create a numeric version while retaining the original value.
    if "DataValue" in combined.columns:
        combined["DataValueRaw"] = combined["DataValue"]

        combined["GDPValue"] = pd.to_numeric(
            combined["DataValue"]
            .astype("string")
            .str.replace(",", "", regex=False)
            .replace(
                {
                    "": pd.NA,
                    "(NA)": pd.NA,
                    "(D)": pd.NA,
                    "--": pd.NA,
                }
            ),
            errors="coerce",
        )

    preferred_columns = [
        "GeoFips",
        "GeoName",
        "LineCode",
        "IndustryName",
        "TimePeriod",
        "Description",
        "CL_UNIT",
        "UNIT_MULT",
        "DataValueRaw",
        "GDPValue",
        "DataValue",
        "Code",
        "TableName",
    ]

    ordered_columns = [
        column
        for column in preferred_columns
        if column in combined.columns
    ]

    remaining_columns = [
        column
        for column in combined.columns
        if column not in ordered_columns
    ]

    combined = combined[
        ordered_columns + remaining_columns
    ]

    combined.to_csv(
        output_file,
        index=False,
    )

    return combined